# 03 — Hipótesis longitudinal con datos reales (EMBED)

**El flujo central del TFG (O7/O8), ejecutable de principio a fin.** Para cada
paciente comparamos su exploración previa "normal" (t-1) con una exploración
posterior (t): en los pacientes *incidentes* la posterior es donde se
diagnostica la sospecha/malignidad; en los *control*, todas son normales.
Medimos dos señales topológicas:

- **Anticipación**: descriptores de persistencia de t-1 **sola** — ¿ya había señal *antes* del diagnóstico?
- **Deriva**: distancia *bottleneck* entre los diagramas de t-1 y t — ¿cuánto cambió la topología?

Reutiliza exactamente el mismo código que `scripts/download_embed_longitudinal.py`
y `scripts/embed_longitudinal_evaluation.py` (este notebook es la vista
interactiva; los scripts, la reproducción de un tirón). La parte de
clasificación aislada sobre CBIS-DDSM vive en `02_real_data_pipeline.ipynb`.

## Requisitos previos (una sola vez)

1. **Acceso a EMBED concedido** (es de acceso controlado; pasos en
   `docs/acceso_datasets_longitudinales.pdf`).
2. **Credenciales AWS** en `~/.aws/credentials`, perfil `[emory-embed]`.
   *Nunca* pongas las claves en el notebook ni en ningún fichero del repo.
3. **Tablas de metadatos** descargadas en `data/embed/` (celda 1 lo hace si faltan).
4. Dependencias: `pip install -r requirements.txt` (incluye `boto3` y
   `pylibjpeg`, necesario porque los DICOM de EMBED vienen comprimidos en
   JPEG Lossless).


## 0. Imports y configuración

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from src.topology import backend
import download_embed_longitudinal as dl
import embed_longitudinal_evaluation as ev

EMBED_DIR = ROOT / "data" / "embed"
print("Backend TDA activo:", backend())


## 1. Tablas de metadatos clínicos

Dos CSV pequeños (frente a los TB de imágenes): `clinical.csv` tiene una fila
por hallazgo con el BI-RADS (`asses`) de cada exploración, y
`metadata_reduced.csv` la ruta S3 de cada DICOM. Esta celda los descarga solo
si no están ya en `data/embed/`.

In [ ]:
import boto3

EMBED_DIR.mkdir(parents=True, exist_ok=True)
TABLAS = {
    "clinical.csv": "tables/EMBED_OpenData_clinical.csv",
    "metadata_reduced.csv": "tables/EMBED_OpenData_metadata_reduced.csv",
}
faltan = {local: remota for local, remota in TABLAS.items() if not (EMBED_DIR / local).exists()}
if faltan:
    s3 = boto3.Session(profile_name="emory-embed", region_name="us-west-2").client("s3")
    for local, remota in faltan.items():
        print(f"Descargando {remota}...")
        s3.download_file("embed-dataset-open", remota, str(EMBED_DIR / local))
else:
    print("Tablas ya descargadas.")


## 2. Estructura longitudinal de la cohorte

¿Cuántos pacientes tienen 2+ exploraciones? ¿Cuántos son candidatos
*incidentes* (primera exploración Normal/Benigna → posterior
Sospechosa/Maligna) y cuántos *control*? Reutiliza
`dl._load_per_study_assessment` y `dl.select_pairs`.

In [ ]:
per_study = dl._load_per_study_assessment(EMBED_DIR / "clinical.csv")

exams = per_study.groupby("empi_anon").size()
print(f"{per_study['empi_anon'].nunique()} pacientes, {len(per_study)} exploraciones")
print(f"{(exams >= 2).sum()} pacientes con 2+ exploraciones")

fig, ax = plt.subplots(figsize=(6, 3))
exams.value_counts().sort_index().plot.bar(ax=ax, color="#1d5a99")
ax.set_xlabel("exploraciones por paciente"); ax.set_ylabel("pacientes")
ax.set_title("Estructura longitudinal de EMBED")
plt.tight_layout(); plt.show()


## 3. Selección del piloto (incidentes vs. control)

`dl.select_pairs` aplica la definición de O7/O8 sobre el BI-RADS por estudio
(`N`/`B` = normal/benigno; `S`/`M`/`K` = sospechoso/altamente
sugestivo/malignidad confirmada). Semilla fija → selección reproducible.

In [ ]:
pares = dl.select_pairs(per_study, n_incident=200, n_control=200, seed=42)
print(f"{len(pares)} pacientes seleccionados "
      f"({sum(p['kind'] == 'incidente' for p in pares)} incidentes, "
      f"{sum(p['kind'] == 'control' for p in pares)} control)")
pares[:3]


## 4. Descarga del piloto (reanudable)

Para cada paciente se eligen las dos exploraciones y **la misma vista**
(lateralidad + proyección, solo tipo `2D`) en ambas, para comparar la misma
región anatómica. Los ficheros ya descargados se saltan, así que re-ejecutar
esta celda es barato. (~8 MB por paciente; 100 pacientes ≈ 1 GB.)

> Equivalente por terminal: `python scripts/download_embed_longitudinal.py --n-incident 200 --n-control 200`

In [ ]:
manifest_path = EMBED_DIR / "longitudinal_pilot_manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print(f"Manifiesto existente: {len(manifest)} pares ya descargados "
          f"({(manifest['kind'] == 'incidente').sum()} incidentes, "
          f"{(manifest['kind'] == 'control').sum()} control)")
else:
    print("No hay piloto descargado: ejecuta en una terminal")
    print("  python scripts/download_embed_longitudinal.py --n-incident 200 --n-control 200")
manifest.head()


## 5. Un par real (t-1, t) de cada clase

La mejor intuición de qué estamos midiendo: en el incidente, la exploración
t-1 fue leída como normal y en t hay un hallazgo; en el control, ambas son
normales.

In [ ]:
def _primer(kind):
    fila = manifest[manifest["kind"] == kind].iloc[0]
    return fila, ev._load_and_preprocess(ROOT / fila["path_t0"]), ev._load_and_preprocess(ROOT / fila["path_t1"])

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, kind in enumerate(["incidente", "control"]):
    fila, img_t0, img_t1 = _primer(kind)
    axes[i, 0].imshow(img_t0, cmap="gray"); axes[i, 0].set_title(f"{kind} {fila['empi_anon']}\n t-1 (normal)")
    axes[i, 1].imshow(img_t1, cmap="gray"); axes[i, 1].set_title("t (posterior)")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout(); plt.show()


## 6. Señales topológicas de todo el piloto

Para cada paciente: diagrama de persistencia de t-1 y de t → vector de
anticipación (Persistence Image de t-1) y deriva (distancia bottleneck H0/H1
entre ambos, con `max_points=100`: los diagramas reales tienen miles de
puntos y sin recortar la distancia tarda ~2 min/par en vez de ~0.2 s;
recortar al top-100 por persistencia es la práctica estándar).

⏱️ La primera vez ~1 s/paciente; las features se guardan en
`data/embed/features_cache.pkl` y las siguientes ejecuciones son instantáneas.

In [ ]:
X_anticipacion, X_deriva, y = ev.compute_features(manifest, n_bins=6, verbose=False)
X_combinada = np.hstack([X_anticipacion, X_deriva])
print("anticipacion:", X_anticipacion.shape, " deriva:", X_deriva.shape)


## 7. ¿Separan las señales a incidentes de controles?

AUC por validación cruzada (5-fold) con Random Forest, igual que
`scripts/embed_longitudinal_evaluation.py`. Con ~100 pacientes esto sigue
siendo un **piloto**: mira las desviaciones típicas antes de sacar
conclusiones.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

resultados = {}
for nombre, X in [("anticipación", X_anticipacion), ("deriva", X_deriva), ("combinada", X_combinada)]:
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X, y, cv=5, scoring="roc_auc")
    resultados[nombre] = (s.mean(), s.std())
    print(f"{nombre:12s}: AUC = {s.mean():.3f} +/- {s.std():.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
nombres = list(resultados)
medias = [resultados[n][0] for n in nombres]
stds = [resultados[n][1] for n in nombres]
ax.bar(nombres, medias, yerr=stds, capsize=4, color=["#888888", "#1d5a99", "#5a991d"])
ax.axhline(0.5, color="k", ls="--", lw=0.8, label="azar")
ax.set_ylabel("AUC (5-fold CV)"); ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.show()


## 8. La distribución de la deriva, a ojo

Si la deriva separa, los incidentes deberían tener distancias t-1→t mayores
que los controles.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for etiqueta, color in [(1, "#b03030"), (0, "#1d5a99")]:
    vals = X_deriva[y == etiqueta, 1]  # distancia H1
    ax.hist(vals, bins=15, alpha=0.6, color=color,
            label="incidente" if etiqueta else "control")
ax.set_xlabel("distancia bottleneck H1 (t-1 → t)"); ax.set_ylabel("pacientes"); ax.legend()
plt.tight_layout(); plt.show()


## 9. Hipótesis secundarias: covariables clínicas por paciente

El AUC global no es el final de la historia. EMBED trae covariables que
permiten **más casos de estudio**: edad (`age_at_study`), densidad mamaria
BI-RADS (`tissueden`) y las fechas de ambas exploraciones (el intervalo
t-1 → t). El manifiesto guarda el `acc_anon` de ambas exploraciones, así que
basta con cruzarlo con `clinical.csv`.

In [ ]:
cov = manifest[["empi_anon", "kind", "acc_t0", "acc_t1"]].copy()
cov["intervalo_dias"] = ev.intervals_days(manifest)

clin = pd.read_csv(EMBED_DIR / "clinical.csv", usecols=["empi_anon", "acc_anon", "age_at_study", "tissueden"])
clin = clin.drop_duplicates(["empi_anon", "acc_anon"]).set_index(["empi_anon", "acc_anon"])
claves_t0 = list(zip(cov["empi_anon"], cov["acc_t0"]))
cov["edad_t0"] = clin["age_at_study"].reindex(claves_t0).values
cov["densidad_t0"] = clin["tissueden"].reindex(claves_t0).values
cov[["empi_anon", "kind", "intervalo_dias", "edad_t0", "densidad_t0"]].head()


## 10. H2 — ¿Es la edad un factor de confusión?

Si los incidentes fueran sistemáticamente mayores que los controles, parte
de cualquier señal podría ser "edad" y no "topología". Comparamos las
distribuciones (Mann-Whitney) y reajustamos la deriva por edad con una
regresión logística.

In [ ]:
from scipy import stats
import statsmodels.api as sm

ok = cov["edad_t0"].notna().values
edad_inc = cov.loc[ok & (y == 1), "edad_t0"]
edad_ctl = cov.loc[ok & (y == 0), "edad_t0"]
u, p_edad = stats.mannwhitneyu(edad_inc, edad_ctl)
print(f"Edad en t-1: incidentes {edad_inc.mean():.1f} ± {edad_inc.std():.1f} vs "
      f"controles {edad_ctl.mean():.1f} ± {edad_ctl.std():.1f}  (Mann-Whitney p={p_edad:.4f})")

X_log = sm.add_constant(pd.DataFrame({
    "deriva_h1": X_deriva[ok, 1],
    "edad": cov.loc[ok, "edad_t0"].values,
}).astype(float))
modelo = sm.Logit(y[ok], X_log).fit(disp=0)
print(modelo.summary2().tables[1].round(4))
print("\nSi el coeficiente de deriva_h1 sigue siendo significativo con la edad en el"
      "\nmodelo, la señal de deriva no se explica solo por la edad.")


## 11. H3 — ¿Depende la señal de la densidad mamaria?

La densidad alta (BI-RADS C/D) dificulta la lectura radiológica. ¿También
degrada la señal topológica de deriva? AUC por subgrupo de densidad
(donde haya n suficiente).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

print("AUC de deriva por densidad mamaria (solo subgrupos con >=15 pacientes y ambas clases):")
for dens, grupo in cov.groupby("densidad_t0"):
    idx = grupo.index.values
    y_g = y[idx]
    if len(idx) < 15 or len(np.unique(y_g)) < 2:
        print(f"  densidad {dens}: n={len(idx)} (insuficiente)")
        continue
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X_deriva[idx], y_g, cv=3, scoring="roc_auc")
    print(f"  densidad {dens}: n={len(idx)}  AUC={s.mean():.3f} +/- {s.std():.3f}")


## 12. H4 — ¿Más deriva cuanto más tiempo pasa? ¿Y la anticipación, cuanto más cerca del diagnóstico?

Dos preguntas temporales:
- En **controles**, la deriva debería crecer suavemente con el intervalo
  (envejecimiento del tejido). En **incidentes**, debería estar dominada por
  la lesión, no por el tiempo.
- La pregunta clínica de oro: en incidentes, ¿la exploración t-1 más
  **cercana** al diagnóstico muestra más señal de anticipación?

In [ ]:
for etiqueta, nombre in [(0, "controles"), (1, "incidentes")]:
    idx = (y == etiqueta) & cov["intervalo_dias"].notna().values
    rho, p = stats.spearmanr(cov.loc[idx, "intervalo_dias"], X_deriva[idx, 1])
    print(f"deriva H1 vs intervalo ({nombre}): Spearman rho={rho:+.3f} (p={p:.4f}, n={idx.sum()})")

# anticipación por cercanía al diagnóstico (solo incidentes)
inc = (y == 1) & cov["intervalo_dias"].notna().values
mediana = cov.loc[inc, "intervalo_dias"].median()
cerca = inc & (cov["intervalo_dias"] <= mediana).values
lejos = inc & (cov["intervalo_dias"] > mediana).values
ctrl = (y == 0)

from sklearn.model_selection import cross_val_score
for nombre, mask in [("t-1 CERCA del diagnostico", cerca), ("t-1 LEJOS del diagnostico", lejos)]:
    idx = np.where(mask | ctrl)[0]
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X_anticipacion[idx], y[idx], cv=5, scoring="roc_auc")
    print(f"anticipacion, {nombre} (mediana={mediana:.0f} dias): AUC={s.mean():.3f} +/- {s.std():.3f} (n_inc={mask.sum()})")


## 13. H5 — ¿Qué dimensión de homología lleva la señal: H0 o H1?

La deriva tiene dos componentes: distancia bottleneck en H0 (componentes
conexas) y en H1 (ciclos/agujeros). ¿Cuál separa mejor?

In [ ]:
for nombre, col in [("solo H0", [0]), ("solo H1", [1]), ("H0+H1", [0, 1])]:
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X_deriva[:, col], y, cv=5, scoring="roc_auc")
    print(f"deriva {nombre:8s}: AUC={s.mean():.3f} +/- {s.std():.3f}")

u, p = stats.mannwhitneyu(X_deriva[y == 1, 1], X_deriva[y == 0, 1])
print(f"\nMann-Whitney deriva H1 incidentes vs controles: p={p:.2e}")


## 14. H6 — El confusor del intervalo (caso de estudio metodológico)

La H4 mostró que la deriva crece con el tiempo transcurrido **en ambos
grupos**. Eso solo es inocuo si los dos grupos tienen intervalos parecidos.
No los tienen: en el diseño original, para los controles se usan sus dos
**primeras** exploraciones (consecutivas, ~1 año), y para los incidentes la
primera normal y la primera maligna (a menudo años después). El tiempo
transcurrido podría estar haciendo parte del trabajo que atribuimos a la
topología.

Tres comprobaciones:
1. Cuánto difieren los intervalos.
2. Qué AUC da **el intervalo solo**, sin ninguna imagen (si es alto, la
   alarma está justificada).
3. Rehacer el análisis con el **diseño emparejado**: el t de cada control se
   re-elige para que su intervalo siga la distribución de los incidentes
   (`python scripts/download_embed_longitudinal.py --match-interval`).

In [ ]:
dias = cov["intervalo_dias"].values
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(dias[y == 1], bins=25, alpha=0.6, color="#b03030", label="incidente")
ax.hist(dias[y == 0], bins=25, alpha=0.6, color="#1d5a99", label="control (diseño original)")
ax.set_xlabel("intervalo t-1 → t (días)"); ax.set_ylabel("pacientes"); ax.legend()
plt.tight_layout(); plt.show()
print(f"mediana: incidentes {np.median(dias[y == 1]):.0f} d, controles {np.median(dias[y == 0]):.0f} d "
      f"(Mann-Whitney p={stats.mannwhitneyu(dias[y == 1], dias[y == 0]).pvalue:.1e})")

rf = lambda: RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
s = cross_val_score(rf(), dias[:, None], y, cv=5, scoring="roc_auc")
print(f"\nAUC del INTERVALO SOLO (sin imagen): {s.mean():.3f} +/- {s.std():.3f}")


### Diseño emparejado por intervalo

Mismos incidentes; mismas exploraciones t-1 de los controles; solo cambia el t
de los controles. Si la señal de deriva sobrevive con los intervalos
igualados, es topología; si se desploma, era tiempo.

In [ ]:
manifest_m = pd.read_csv(EMBED_DIR / "longitudinal_pilot_manifest_matched.csv")
Xa_m, Xd_m, y_m = ev.compute_features(manifest_m, n_bins=6, verbose=False)
dias_m = ev.intervals_days(manifest_m)
print(f"Emparejado: {len(manifest_m)} pacientes ({(y_m == 1).sum()} inc, {(y_m == 0).sum()} ctl); "
      f"mediana intervalo inc {np.median(dias_m[y_m == 1]):.0f} d vs ctl {np.median(dias_m[y_m == 0]):.0f} d "
      f"(Mann-Whitney p={stats.mannwhitneyu(dias_m[y_m == 1], dias_m[y_m == 0]).pvalue:.2f})")

comparacion = {}
for etiqueta, (Xa, Xd, yy, dd) in {"original": (X_anticipacion, X_deriva, y, dias),
                                    "emparejado": (Xa_m, Xd_m, y_m, dias_m)}.items():
    for señal, X in [("intervalo solo", dd[:, None]), ("anticipación", Xa),
                     ("deriva", Xd), ("combinada", np.hstack([Xa, Xd]))]:
        s = cross_val_score(rf(), X, yy, cv=5, scoring="roc_auc")
        comparacion[(señal, etiqueta)] = (s.mean(), s.std())

señales = ["intervalo solo", "anticipación", "deriva", "combinada"]
tabla = pd.DataFrame({e: [f"{comparacion[(s, e)][0]:.3f} ± {comparacion[(s, e)][1]:.3f}" for s in señales]
                      for e in ["original", "emparejado"]}, index=señales)
display(tabla)

fig, ax = plt.subplots(figsize=(7, 3.2))
x = np.arange(len(señales)); w = 0.38
for k, (e, color) in enumerate([("original", "#999999"), ("emparejado", "#1d5a99")]):
    ax.bar(x + (k - 0.5) * w, [comparacion[(s, e)][0] for s in señales], w,
           yerr=[comparacion[(s, e)][1] for s in señales], capsize=3, color=color, label=e)
ax.axhline(0.5, color="k", ls="--", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(señales); ax.set_ylim(0, 1); ax.set_ylabel("AUC (5-fold CV)")
ax.legend(); plt.tight_layout(); plt.show()


## 15. Casos de estudio extremos

El incidente con más deriva (la señal en su versión más clara) y el control
con más deriva (el "falso positivo" más peligroso del piloto: tejido que
cambió mucho sin malignidad).

In [ ]:
extremos = [
    ("incidente con MAS deriva", int(np.where(y == 1)[0][np.argmax(X_deriva[y == 1, 1])])),
    ("control con MAS deriva",   int(np.where(y == 0)[0][np.argmax(X_deriva[y == 0, 1])])),
]
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, (titulo, idx) in enumerate(extremos):
    fila = manifest.iloc[idx]
    axes[i, 0].imshow(ev._load_and_preprocess(ROOT / fila["path_t0"]), cmap="gray")
    axes[i, 0].set_title(f"{titulo}\n{fila['empi_anon']} — t-1")
    axes[i, 1].imshow(ev._load_and_preprocess(ROOT / fila["path_t1"]), cmap="gray")
    axes[i, 1].set_title(f"t  (deriva H1 = {X_deriva[idx, 1]:.3f})")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout(); plt.show()


## Lectura de los resultados y siguientes pasos

Con el piloto de 99 pacientes (resultados de referencia en
`results/embed_longitudinal_pilot.txt` y en la memoria, cap. 7):

- **Deriva ≈ 0.80 ± 0.06** — el cambio topológico entre t-1 y t es una señal
  real y robusta. Esperable: en t ya hay una lesión.
- **Anticipación ≈ 0.55 ± 0.10** — cerca del azar. La pregunta valiosa
  ("¿ya se veía antes?") sigue abierta: o el precursor es demasiado sutil
  para este descriptor (vista completa a 224×224), o no hay señal.

Siguientes pasos:
- Ampliar el piloto (`--n-incident 100 --n-control 100`...): hay 644
  incidentes y 7256 controles candidatos (issue #95).
- Probar descriptores más sensibles para la anticipación (mayor resolución,
  localización de la región sospechosa, otra filtración: ver
  `--filtration distance_transform` en `run_pipeline.py`).
